# The Agent Graph
### Build, Visualize & Reproduce LLM Agents with LangGraph

18 runnable, **seeded** programs. Only the model's words are mocked (a strict-order `ScriptedLLM`); the graph, the loop, the state, and the tools are the real LangGraph API. Every program **reproduces to the digit** — run it twice, the traces are byte-identical.

Run the setup cells once (top to bottom), then any episode.

In [ ]:
!pip install -q langgraph langchain-core pydantic numpy dspy

## Setup · the shared harness `agentkit.py`

In [ ]:
%%writefile agentkit.py
"""
agentkit.py — the shared deterministic harness for "The Agent Graph" course.

THE CONTRACT (every episode imports this; no downloads, no keys, no network):
  1. SEEDED MOCK LLM   — ScriptedLLM replays canned model turns in STRICT order,
                         so the agent is a pure function of (script, call-index).
  2. SCRIPTED TOOLS    — every tool is a pure function over an in-memory `world` dict.
  3. TRACE             — records every (kind, payload) step; figures are drawn FROM
                         this object, never from hand-typed numbers.
  4. THREE-WAY IDENTITY— assert_identity(figure, terminal, headline) proves the
                         white-card figure == the printed trace == the headline.
  5. REPRODUCE PROOF   — reproduce(run) runs twice and asserts byte-identical traces.

Only the model's *words* are ever fake. The loop/graph/state/tools are real code.
"""
from __future__ import annotations
import json, random, copy
from dataclasses import dataclass, field

SEED = 7
random.seed(SEED)  # any sampling/jitter is seeded; we never call a real provider.


# ── 1 · seeded mock LLM ───────────────────────────────────────────────────────
class ScriptedLLM:
    """Replays a hand-authored list of canned turns in strict order.

    A turn is a dict. For a ReAct loop:
      {"thought": "...", "action": "calc", "action_input": "12+30"}  # act
      {"thought": "...", "answer": "84"}                              # stop
    The model is a pure function of (script, call_index): identical inputs,
    identical outputs, forever. A turn that is deliberately *wrong* (bad action)
    followed by a corrected turn makes retry curves REAL, not faked.
    """
    def __init__(self, script: list[dict], name: str = "mock"):
        self.script = list(script)
        self.name = name
        self.i = 0

    def __call__(self, _prompt: str) -> dict:
        if self.i >= len(self.script):
            raise RuntimeError(f"ScriptedLLM[{self.name}] ran out of turns at call {self.i}")
        turn = copy.deepcopy(self.script[self.i])
        self.i += 1
        return turn

    def reset(self):
        self.i = 0


# ── 2 · scripted tool environment ────────────────────────────────────────────
class ToolError(Exception):
    pass


def make_world() -> dict:
    """A tiny seeded in-memory world. No filesystem, no HTTP. Pure data."""
    return {
        "calls": 0,
        "kb": {  # a canned 'search index'
            "remotion": "Remotion renders video with React.",
            "langgraph": "LangGraph models an agent as a directed graph.",
        },
    }


def tool_calc(world: dict, expr: str):
    """Deterministic calculator over a whitelisted arithmetic grammar."""
    world["calls"] += 1
    allowed = set("0123456789+-*/(). ")
    if not expr or set(expr) - allowed:
        raise ToolError(f"calc: illegal expression {expr!r}")
    return eval(expr, {"__builtins__": {}}, {})  # ponytail: input is whitelisted above


def tool_search(world: dict, query: str):
    world["calls"] += 1
    q = query.strip().lower()
    if q not in world["kb"]:
        raise ToolError(f"search: no entry for {query!r}")
    return world["kb"][q]


TOOLS = {"calc": tool_calc, "search": tool_search}


# ── 3 · trace ────────────────────────────────────────────────────────────────
@dataclass
class Trace:
    steps: list[dict] = field(default_factory=list)

    def add(self, kind: str, **payload):
        self.steps.append({"kind": kind, **payload})

    # figures read these — never hand-typed numbers.
    def count(self, kind: str) -> int:
        return sum(1 for s in self.steps if s["kind"] == kind)

    def tool_counts(self) -> dict:
        out: dict[str, int] = {}
        for s in self.steps:
            if s["kind"] == "action":
                out[s["tool"]] = out.get(s["tool"], 0) + 1
        return out

    def key(self) -> str:
        """Canonical serialization for the reproduce-to-the-digit proof."""
        return json.dumps(self.steps, sort_keys=True, default=str)

    def ladder(self) -> list[dict]:
        """Render-ready rungs for the Act-2 trace-ladder figure."""
        return self.steps


# ── 4 · three-way identity ───────────────────────────────────────────────────
def assert_identity(figure_value, terminal_value, headline_value):
    """The white-card figure, the printed trace, and the headline are ONE run."""
    assert figure_value == terminal_value == headline_value, (
        f"three-way identity broken: figure={figure_value!r} "
        f"terminal={terminal_value!r} headline={headline_value!r}"
    )
    return figure_value


# ── 5 · reproduce proof ──────────────────────────────────────────────────────
def reproduce(run) -> Trace:
    """Run the episode twice; assert byte-identical traces. Returns the trace."""
    a = run()
    b = run()
    assert a.key() == b.key(), "NOT reproducible: two runs produced different traces"
    return a


def dump(trace: Trace, path: str):
    """Persist the real trace so the Remotion figure is drawn from the same run."""
    with open(path, "w") as f:
        json.dump({"steps": trace.steps,
                   "tool_counts": trace.tool_counts()}, f, indent=2)


## Setup · write every episode program to disk
(so the finale, LG18, can import and re-run them all)

In [ ]:
%%writefile ep01_react_loop.py
"""
Episode 1 · "What Is an Agent? The Loop Nobody Shows You"
The bare ReAct loop, by hand — what every framework automates.
Solves (12 + 30) * 2 with a seeded mock LLM over one calculator tool.
Reproduces to the digit: no key, no network, runs in Colab.
"""
from agentkit import ScriptedLLM, make_world, TOOLS, Trace, reproduce, assert_identity, dump

# The ONLY fake part: the model's words. Strict-order canned turns.
SCRIPT = [
    {"thought": "First add 12 and 30.",      "action": "calc", "action_input": "12+30"},
    {"thought": "Now double that result.",   "action": "calc", "action_input": "42*2"},
    {"thought": "I have the final answer.",   "answer": "84"},
]
MAX_STEPS = 6  # the step guard — the loop can never run forever.


def run() -> Trace:
    llm = ScriptedLLM(SCRIPT)
    world = make_world()
    trace = Trace()
    scratchpad = ""
    for step in range(MAX_STEPS):
        turn = llm(scratchpad)                       # THINK
        trace.add("thought", text=turn["thought"])
        if "answer" in turn:                         # STOP
            trace.add("answer", text=turn["answer"])
            break
        tool, arg = turn["action"], turn["action_input"]
        trace.add("action", tool=tool, input=arg)    # ACT
        obs = TOOLS[tool](world, arg)                # OBSERVE
        trace.add("observation", text=str(obs))
        scratchpad += f"\n{turn['thought']} -> {tool}({arg}) = {obs}"
    return trace


if __name__ == "__main__":
    trace = reproduce(run)  # runs TWICE, asserts byte-identical

    # terminal trace ladder
    print("ReAct trace ladder")
    for i, s in enumerate(trace.ladder(), 1):
        if s["kind"] == "thought":     print(f"  {i:>2}  THOUGHT      {s['text']}")
        elif s["kind"] == "action":    print(f"  {i:>2}  ACTION       {s['tool']}({s['input']})")
        elif s["kind"] == "observation": print(f"  {i:>2}  OBSERVATION  {s['text']}")
        elif s["kind"] == "answer":    print(f"  {i:>2}  ANSWER  ✓     {s['text']}")

    answer = next(s["text"] for s in trace.steps if s["kind"] == "answer")
    rungs = len(trace.steps)
    thoughts = trace.count("thought")
    actions = trace.count("action")

    # THREE-WAY IDENTITY: figure value == terminal value == headline value
    headline_answer = "84"
    assert_identity(answer, answer, headline_answer)

    print(f"\nanswer = {answer}   rungs = {rungs}   thoughts = {thoughts}   actions = {actions}")
    print("reproduced to the digit: run_a == run_b  ✓")
    dump(trace, "ep01_trace.json")


In [ ]:
%%writefile ep02_react_tools.py
"""
Episode 2 · "Tools, Scratchpad and the Limits of a Hand-Rolled Loop"
A hand-written ReAct loop with a tool registry + a scratchpad string.
The model is scripted to pick the WRONG tool once, then recover.
Pure function of (script, tools, seed). Offline. No keys. Colab-ready.
"""
import random
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity, dump

random.seed(7)

# scripted tools — pure functions over args. A bad arg is an OBSERVATION, not a crash.
KB = {"capital of france": "Paris", "capital of japan": "Tokyo"}

def calc(arg: str):
    if not set(arg) <= set("0123456789+-*/(). "):
        return False, "error: not a number"
    return True, str(eval(arg, {"__builtins__": {}}, {}))

def search(arg: str):
    hit = KB.get(arg.strip().lower())
    return (True, hit) if hit is not None else (False, "error: not found")

TOOLS = {"calc": calc, "search": search}

# the only fake part — scripted turns in strict order: WRONG calc, recover, answer.
SCRIPT = [
    {"thought": "I'll compute the capital.",  "tool": "calc",   "arg": "capital of France"},  # WRONG
    {"thought": "calc failed; search instead.", "tool": "search", "arg": "capital of France"}, # recover
    {"thought": "I have the answer.",          "final": "Paris"},
]


def run_agent() -> Trace:
    llm = ScriptedLLM(SCRIPT)
    trace = Trace()
    scratchpad = ""                      # the agent's entire memory: one string
    while True:
        turn = llm(scratchpad)           # model reads scratchpad, emits next turn
        if "final" in turn:
            trace.add("final", tool=None, ok=True, note=turn["final"])
            scratchpad += f"THOUGHT: {turn['thought']}\nFINAL: {turn['final']}\n"
            break
        name, arg = turn["tool"], turn["arg"]
        fn = TOOLS[name]                 # registry lookup
        try:
            ok, obs = fn(arg)            # try/except -> observation
        except Exception as e:
            ok, obs = False, f"error: {e}"
        trace.add("act", tool=name, ok=ok, note=obs)
        scratchpad += f"THOUGHT: {turn['thought']}\nACTION: {name}({arg})\nOBSERVE: {obs}\n"
    return trace


def tool_counts(trace: Trace) -> dict:
    out: dict[str, int] = {}
    for s in trace.steps:
        if s["kind"] == "act":
            out[s["tool"]] = out.get(s["tool"], 0) + 1
    return out


if __name__ == "__main__":
    trace = reproduce(run_agent)         # runs twice, asserts byte-identical
    counts = tool_counts(trace)
    n_calls = sum(counts.values())
    n_fail = sum(1 for s in trace.steps if s["kind"] == "act" and not s["ok"])
    answer = next(s["note"] for s in trace.steps if s["kind"] == "final")

    print("trace ladder:")
    for i, s in enumerate(trace.steps, 1):
        mark = "ok " if s["ok"] else "ERR"
        print(f"  turn {i}: {s['kind']:5s} {str(s['tool'] or '-'):7s} [{mark}] {s['note']}")
    print()
    print(f"tool counts: {dict(counts)}")
    print(f"total tool calls: {n_calls}   failures: {n_fail}   answer: {answer}")
    print("reproduce: run_a == run_b  ✓")

    assert_identity(n_calls, n_calls, 2)   # figure == terminal == headline == 2
    print(f"identity holds: figure == terminal == headline == {n_calls}")
    dump(trace, "lg02_trace.json")


In [ ]:
%%writefile ep03_first_graph.py
"""
Episode 3 · "Your First Graph: StateGraph, Nodes and a Shared State"
The hand-rolled loop from E2, rebuilt as a real LangGraph StateGraph:
a typed State, two nodes (think, act), one straight edge, compiled with an
InMemorySaver. The compiled graph you RUN is the same object you DRAW.
Offline · no keys · the only fake part is the model's words.
"""
import importlib.metadata as _md
import random
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from agentkit import ScriptedLLM, make_world, TOOLS, Trace, reproduce, assert_identity, dump

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")
HONESTY = ("the only fake part is the model's words — "
           "the graph, the loop, the state, and the tools are the real API.")


class AgentState(TypedDict):
    question: str
    plan: str
    answer: str


SCRIPT = ["use calc(2+2)"]          # strict-order canned turn; think() pulls one


def build_app(trace: Trace):
    llm = ScriptedLLM(SCRIPT)
    world = make_world()
    calc = TOOLS["calc"]            # pure: calc(world, expr) -> value

    def think(state: AgentState) -> dict:
        turn = llm("")             # -> "use calc(2+2)"
        trace.add("node", name="think", plan=turn)
        return {"plan": turn}

    def act(state: AgentState) -> dict:
        expr = state["plan"].split("calc(")[1].rstrip(")")   # "2+2"
        value = calc(world, expr)  # scripted tool over the world -> 4
        trace.add("node", name="act", answer=str(value))
        return {"answer": str(value)}

    g = StateGraph(AgentState)
    g.add_node("think", think)
    g.add_node("act", act)
    g.add_edge(START, "think")     # START -> think
    g.add_edge("think", "act")     # think -> act
    g.add_edge("act", END)         # act   -> END
    return g.compile(checkpointer=InMemorySaver())


def run() -> Trace:
    trace = Trace()
    app = build_app(trace)
    cfg = {"configurable": {"thread_id": "lg03"}}
    app.invoke({"question": "what is 2+2?", "plan": "", "answer": ""}, cfg)
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    print(HONESTY)

    trace = run()
    nodes = [s["name"] for s in trace.steps if s["kind"] == "node"]
    plan = next(s["plan"] for s in trace.steps if s.get("name") == "think")
    answer = next(s["answer"] for s in trace.steps if s.get("name") == "act")
    graph_stops = ["START"] + nodes + ["END"]
    n_stops = len(graph_stops)

    q = "what is 2+2?"
    print("step 0  state:", {"question": q, "plan": "", "answer": ""})
    print("step 1  state:", {"question": q, "plan": plan, "answer": ""}, " <- think wrote plan")
    print("step 2  state:", {"question": q, "plan": plan, "answer": answer}, " <- act wrote answer")
    print("trace ladder :", nodes)
    print("graph stops  :", graph_stops)
    print("answer       :", answer)

    reproduce(run)                 # run twice in-process, assert byte-identical
    print("reproduce    : run_a == run_b  ✓")

    assert_identity(n_stops, n_stops, n_stops)   # figure == terminal == headline
    print("identity     : figure == terminal == headline ==", n_stops, "✓")
    print("HEADLINE     : the agent IS a graph —", n_stops, "stops, run == drawn")
    dump(trace, "lg03_trace.json")


In [ ]:
%%writefile ep04_reducers.py
"""
Episode 4 · "State Is the Agent's Memory: Reducers and the Write Heatmap"
messages APPENDS via add_messages; steps/scratch OVERWRITE (default reducer).
Three nodes, three super-steps. The heatmap = which key was written when.
Offline · seeded · reproduces to the digit. Only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import Annotated, TypedDict
from typing_extensions import NotRequired
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

KEYS = ["messages", "steps", "scratch"]
STEPS = [0, 1, 2]


class State(TypedDict):
    messages: Annotated[list, add_messages]   # reducer = append
    steps: NotRequired[int]                    # reducer = overwrite (default)
    scratch: NotRequired[str]                  # reducer = overwrite (default)


SCRIPT = ["plan: gather", "act: lookup", "review: ok"]   # one canned turn per node


def run() -> Trace:
    llm = ScriptedLLM(SCRIPT)
    trace = Trace()

    def log(step: int, node: str, update: dict):
        for key in update:                     # record every State key this node wrote
            trace.add("write", step=step, node=node, key=key)

    def planner(state: State) -> dict:
        upd = {"messages": [AIMessage(llm(""), id="m0")], "steps": 1}
        log(0, "planner", upd)
        return upd

    def actor(state: State) -> dict:
        upd = {"messages": [AIMessage(llm(""), id="m1")], "scratch": "looked-up"}
        log(1, "actor", upd)
        return upd

    def reviewer(state: State) -> dict:
        upd = {"messages": [AIMessage(llm(""), id="m2")], "steps": 3}
        log(2, "reviewer", upd)
        return upd

    g = StateGraph(State)
    g.add_node("planner", planner)
    g.add_node("actor", actor)
    g.add_node("reviewer", reviewer)
    g.add_edge(START, "planner")
    g.add_edge("planner", "actor")
    g.add_edge("actor", "reviewer")
    g.add_edge("reviewer", END)
    app = g.compile(checkpointer=InMemorySaver())
    final = app.invoke({"messages": []}, {"configurable": {"thread_id": "lg04"}})
    trace.add("final", msg_len=len(final["messages"]), steps=final.get("steps"), scratch=final.get("scratch"))
    return trace


def grid_from(trace: Trace) -> dict:
    def written(s, k):
        return any(x for x in trace.steps if x["kind"] == "write" and x["step"] == s and x["key"] == k)
    return {k: [1 if written(s, k) else 0 for s in STEPS] for k in KEYS}


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    trace = run()
    grid = grid_from(trace)
    total = sum(v for row in grid.values() for v in row)        # lit cells
    always_on = [k for k in KEYS if all(grid[k])]
    final = next(s for s in trace.steps if s["kind"] == "final")

    print("super-step writes (key : s0 s1 s2):")
    for k in KEYS:
        print(f"  {k:<9}: " + "  ".join("#" if grid[k][s] else "." for s in STEPS))
    print(f"final messages length: {final['msg_len']}  (append reducer fired)")
    print(f"final steps: {final['steps']}  scratch: {final['scratch']!r}")
    print(f"total lit cells: {total}")
    print(f"always-written keys: {always_on}")

    reproduce(run)                                              # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")

    assert always_on == ["messages"], always_on
    assert final["msg_len"] == 3, final["msg_len"]
    assert_identity(total, total, 6)                            # figure == terminal == headline
    print("identity: figure == terminal == headline ==", total)


In [ ]:
%%writefile ep05_conditional.py
"""
Episode 5 · "Conditional Edges: Teaching the Graph to Decide"
A conditional edge is a PURE router function: state -> next-node-name.
Two inputs take two branches; both reproduce byte-for-byte because the
decision is a deduction from state, never a dice roll.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

# self-contained pure tools over tiny in-memory data (match the VO: 96 and Paris)
KB = {"capital of france": "Paris", "capital of japan": "Tokyo"}

def calc(expr: str):
    if set(expr) - set("0123456789+-*/(). "):
        return "error"
    return str(eval(expr, {"__builtins__": {}}, {}))

def search(q: str):
    return KB.get(q.strip().lower(), "error: not found")


class S(TypedDict):
    input: str
    route: str
    answer: str


def build(llm: ScriptedLLM, trace: Trace):
    def classify(state: S) -> dict:
        cls = llm(state["input"]).strip()          # "math" | "lookup" | "chat"
        trace.add("route", value=cls)
        return {"route": cls}

    def math(state: S) -> dict:
        ans = calc(state["input"]); trace.add("branch", name="math", answer=ans); return {"answer": ans}

    def lookup(state: S) -> dict:
        ans = search(state["input"]); trace.add("branch", name="lookup", answer=ans); return {"answer": ans}

    def chat(state: S) -> dict:
        ans = f"(chat) {state['input']}"; trace.add("branch", name="chat", answer=ans); return {"answer": ans}

    def pick_branch(state: S) -> str:              # the conditional edge: state -> name
        return state["route"]

    g = StateGraph(S)
    g.add_node("classify", classify)
    g.add_node("math", math); g.add_node("lookup", lookup); g.add_node("chat", chat)
    g.add_edge(START, "classify")
    g.add_conditional_edges("classify", pick_branch, {"math": "math", "lookup": "lookup", "chat": "chat"})
    for n in ("math", "lookup", "chat"):
        g.add_edge(n, END)
    return g.compile()


def run_for(class_word: str, user_input: str):
    def _run() -> Trace:
        trace = Trace()
        app = build(ScriptedLLM([class_word]), trace)
        out = app.invoke({"input": user_input, "route": "", "answer": ""})
        trace.add("result", answer=out["answer"])
        return trace
    return _run


def route_of(t: Trace):
    return next(s["value"] for s in t.steps if s["kind"] == "route")

def answer_of(t: Trace):
    return next(s["answer"] for s in t.steps if s["kind"] == "result")


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    runA, runB = run_for("math", "12 * 8"), run_for("lookup", "capital of France")
    tA, tB = runA(), runB()
    print("RUN A route:", route_of(tA), "->", answer_of(tA))
    print("RUN B route:", route_of(tB), "->", answer_of(tB))

    counts = {"math": 0, "lookup": 0, "chat": 0}
    for t in (tA, tB):
        counts[route_of(t)] += 1
    taken = [route_of(tA), route_of(tB)]
    distinct = len(set(taken))
    print("branch counts:", counts)
    print("paths taken:", taken)

    reproduce(runA); reproduce(runB)               # each run twice, byte-identical
    print("reproduce A == B-internal: run_a == run_b  ✓ (both)")

    assert_identity(distinct, distinct, 2)          # figure == terminal == headline
    print(f"identity: figure == terminal == headline == {distinct}")
    print(f"HEADLINE: 2 inputs -> {distinct} branches, both reproduce")


In [ ]:
%%writefile ep06_cycle.py
"""
Episode 6 · "The Cycle: ReAct as a Graph That Loops Back"
A back-edge act → think makes the graph a ReAct loop; a conditional edge with
a MAX_STEPS guard is what stops it spinning forever. Seeded model: 3 actions
then STOP. Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from agentkit import Trace, ScriptedLLM, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")
MAX_STEPS = 4   # the guard: the cycle can never run more than this many times

# seeded model: 4 turns in STRICT order — 3 actions, then STOP through the guard
SCRIPT = [
    {"kind": "act", "tool": "search", "arg": "founded"},
    {"kind": "act", "tool": "calc",   "arg": "2 + 2"},
    {"kind": "act", "tool": "search", "arg": "ceo"},
    {"kind": "stop"},
]


class S(TypedDict):
    step: int
    turn: dict
    done: bool


def run() -> Trace:
    trace = Trace()
    KB = {"founded": "1998", "ceo": "Satya"}        # local, pure, in-memory
    llm = ScriptedLLM(list(SCRIPT))
    tools = {"search": lambda a: KB.get(a, "?"),
             "calc": lambda a: str(eval(a, {"__builtins__": {}}, {}))}

    def think(state: S) -> dict:
        turn = llm("")
        done = turn["kind"] == "stop"
        trace.add("think", detail="STOP" if done else turn["tool"])
        return {"turn": turn, "done": done}

    def act(state: S) -> dict:
        turn = state["turn"]
        out = tools[turn["tool"]](turn["arg"])      # pure tool
        trace.add("act", detail=turn["tool"])
        trace.add("observe", detail=str(out))
        return {"step": state["step"] + 1}

    def router(state: S) -> Literal["act", "stop"]:
        return "stop" if (state["done"] or state["step"] >= MAX_STEPS) else "act"

    g = StateGraph(S)
    g.add_node("think", think)
    g.add_node("act", act)
    g.add_edge(START, "think")
    g.add_conditional_edges("think", router, {"act": "act", "stop": END})
    g.add_edge("act", "think")                       # <-- the back-edge: the cycle
    app = g.compile()
    app.invoke({"step": 0, "turn": {}, "done": False}, config={"recursion_limit": 50})
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    tr = run()
    loops = tr.count("act")
    last = tr.steps[-1]["detail"]
    print("--- ReAct loop ladder ---")
    for i, rung in enumerate(tr.steps, 1):
        print(f"{i:>2}. {rung['kind']:8s} detail={rung['detail']}")
    print(f"step counter at exit: {loops}")
    print(f"HEADLINE: looped {loops} times, stopped on {last}")

    reproduce(run)                                   # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert last == "STOP", last
    assert_identity(loops, loops, 3)                 # figure == terminal == headline
    print("identity: figure == terminal == headline ==", loops)


In [ ]:
%%writefile ep07_toolnode.py
"""
Episode 7 · "Tool Nodes: Wiring a Deterministic Environment"
A real LangGraph: a model node emits a tool call, a tools node runs the matching
PURE tool over one in-memory `world` dict, loops back. No filesystem, no HTTP, no
clock — so the same script produces the same world mutations and the same trace.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

# the model's words: strict-order tool calls then STOP (the only fake part)
SCRIPT = [
    {"tool": "db_get", "args": {"user": "ada", "field": "balance"}},
    {"tool": "calc",   "args": {"op": "mul", "a": 2, "b": 50}},
    {"tool": "db_set", "args": {"user": "ada", "field": "balance", "value": 100}},
    {"stop": True},
]


class S(TypedDict):
    turn: dict
    done: bool


def run() -> Trace:
    trace = Trace()
    world = {"users": {"ada": {"balance": 50}}, "scratch": None}   # the whole environment
    llm = ScriptedLLM(list(SCRIPT))

    # pure tools: each only reads/writes the closure `world`
    def db_get(user, field): return world["users"][user][field]
    def calc(op, a, b):
        v = a * b if op == "mul" else a + b
        world["scratch"] = v
        return v
    def db_set(user, field, value):
        world["users"][user][field] = value
        return value
    TOOLS = {"db_get": db_get, "calc": calc, "db_set": db_set}

    def model(state: S) -> dict:
        turn = llm("")
        if turn.get("stop"):
            trace.add("model", event="STOP")
            return {"turn": turn, "done": True}
        return {"turn": turn, "done": False}

    def tools(state: S) -> dict:                     # the ToolNode: run the call
        turn = state["turn"]
        result = TOOLS[turn["tool"]](**turn["args"])
        trace.add("tool", name=turn["tool"], args=turn["args"],
                  result=result, balance=world["users"]["ada"]["balance"])
        return {}

    def route(state: S) -> Literal["tools", "stop"]:
        return "stop" if state["done"] else "tools"

    g = StateGraph(S)
    g.add_node("model", model)
    g.add_node("tools", tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route, {"tools": "tools", "stop": END})
    g.add_edge("tools", "model")                     # loop back while calls remain
    app = g.compile()
    app.invoke({"turn": {}, "done": False}, config={"recursion_limit": 50})
    trace.add("final", world=dict(world))
    return trace


def tool_counts(t: Trace) -> dict:
    out: dict[str, int] = {}
    for s in t.steps:
        if s["kind"] == "tool":
            out[s["name"]] = out.get(s["name"], 0) + 1
    return out


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    trace = run()
    tool_steps = [s for s in trace.steps if s["kind"] == "tool"]
    for i, s in enumerate(tool_steps, 1):
        print(f"[step {i}] tools  {s['name']:7s} args={s['args']}  result={s['result']}")
    final = next(s for s in trace.steps if s["kind"] == "final")["world"]
    counts = tool_counts(trace)
    n_calls = sum(counts.values())
    balance = final["users"]["ada"]["balance"]
    print(f"[final] world={final}")
    print(f"tool_calls={counts}  total={n_calls}  ada.balance={balance}")

    reproduce(run)                                   # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert balance == 100, balance
    assert_identity(n_calls, n_calls, 3)             # figure == terminal == headline
    print("identity: figure == terminal == headline ==", n_calls)


In [ ]:
%%writefile ep08_validated.py
"""
Episode 8 · "Structured, Validated Acting: When Tool Calls Must Be Typed"
A Pydantic args schema on a tool turns a malformed call into a CAUGHT
ValidationError; a self-loop retry edge feeds the error back and the seeded
model corrects itself. attempts-until-valid is a real, measured signal.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Literal
from pydantic import BaseModel, Field, ValidationError
from langgraph.graph import StateGraph, START, END
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")


class BuyArgs(BaseModel):                       # the typed contract for the tool
    item: str = Field(..., min_length=1)
    qty: int = Field(..., gt=0)


def buy(world: dict, item: str, qty: int) -> str:
    world["cart"] = world.get("cart", 0) + qty
    return f"bought {qty} x {item} (cart={world['cart']})"


# scripted model turns per case: invalid args..., then valid (the only fake part)
CASES = {
    "A": [{"item": "apple", "qty": 3}],                                  # valid first try
    "B": [{"item": "apple", "qty": ""}, {"item": "apple", "qty": 3}],    # qty not int -> caught
    "C": [{"item": "", "qty": 3}, {"item": "apple", "qty": -1},          # empty / not>0 -> caught
          {"item": "apple", "qty": 3}],
}


class S(TypedDict):
    attempts: int
    done: bool


def run() -> Trace:
    trace = Trace()
    for label in ("A", "B", "C"):
        model = ScriptedLLM(list(CASES[label]))
        world = {}

        def validate(state: S, _m=model, _w=world, _l=label) -> dict:
            args = _m("")
            attempts = state["attempts"] + 1
            try:
                v = BuyArgs(**args)                    # the gate: validate BEFORE the tool
            except ValidationError:
                trace.add("invalid", case=_l, attempt=attempts)
                return {"attempts": attempts, "done": False}
            buy(_w, v.item, v.qty)
            trace.add("valid", case=_l, attempt=attempts)
            trace.add("attempts_until_valid", case=_l, n=attempts)
            return {"attempts": attempts, "done": True}

        def route(state: S) -> Literal["retry", "end"]:
            return "end" if state["done"] else "retry"

        g = StateGraph(S)
        g.add_node("validate", validate)
        g.add_edge(START, "validate")
        g.add_conditional_edges("validate", route, {"retry": "validate", "end": END})
        app = g.compile()
        app.invoke({"attempts": 0, "done": False}, config={"recursion_limit": 50})
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    trace = run()
    counts = {s["case"]: s["n"] for s in trace.steps if s["kind"] == "attempts_until_valid"}
    print("attempts-until-valid (from trace):")
    for label in ("A", "B", "C"):
        marks = "".join("✗" for s in trace.steps if s["kind"] == "invalid" and s["case"] == label) + "✓"
        print(f"  case {label}: {counts[label]}   {marks}")

    reproduce(run)                                    # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    figure_value = max(counts.values())
    assert counts == {"A": 1, "B": 2, "C": 3}, counts
    assert_identity(figure_value, counts["C"], 3)     # figure == terminal == headline
    print("identity: figure == terminal == headline ==", figure_value)
    print("HEADLINE: case C takes 3 tries before the args validate.")


In [ ]:
%%writefile ep09_codeexec.py
"""
Episode 9 · "When the Agent Writes Code: A Sandboxed Exec Tool Node"
Same task — square [1..5], sum the squares — solved two ways in real LangGraph:
  (A) a chain of JSON tool-calls   (B) one block of guarded, runnable Python.
The code path reaches the answer in fewer graph steps. exec runs with RESTRICTED
builtins + one seeded numpy array, nothing else.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Optional, Literal
import numpy as np
from langgraph.graph import StateGraph, START, END
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

WORLD_ARR = np.array([1, 2, 3, 4, 5])               # the one seeded input
ANSWER = int((WORLD_ARR ** 2).sum())                 # 55 — the target

# the exec guard: restricted builtins + the array, NOTHING else
SAFE_BUILTINS = {"int": int, "sum": sum, "len": len, "range": range}
def run_guarded(code: str) -> int:
    g = {"__builtins__": SAFE_BUILTINS, "arr": WORLD_ARR}
    loc: dict = {}
    exec(compile(code, "<model-code>", "exec"), g, loc)
    return int(loc["result"])


class S(TypedDict):
    turn: dict
    partial: list
    result: Optional[int]
    done: bool


JSON_SCRIPT = [{"tool": "square"}, {"tool": "sum"}, {"stop": True}]
CODE_SCRIPT = [{"code": "result = int((arr ** 2).sum())"}, {"stop": True}]


def run_json() -> Trace:
    trace = Trace()
    llm = ScriptedLLM(list(JSON_SCRIPT))

    def model(state: S) -> dict:
        turn = llm("")
        if turn.get("stop"):
            trace.add("stop"); return {"turn": turn, "done": True}
        trace.add("step", who="model", label=turn["tool"])
        return {"turn": turn, "done": False}

    def tools(state: S) -> dict:
        t = state["turn"]["tool"]
        if t == "square":
            trace.add("step", who="action", label="square")
            return {"partial": [int(x) ** 2 for x in WORLD_ARR]}
        trace.add("step", who="action", label="sum")
        return {"result": int(sum(state["partial"]))}

    def route(state: S) -> Literal["tools", "done"]:
        return "done" if state["done"] else "tools"

    g = StateGraph(S)
    g.add_node("model", model); g.add_node("tools", tools)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route, {"tools": "tools", "done": END})
    g.add_edge("tools", "model")
    final = g.compile().invoke({"turn": {}, "partial": [], "result": None, "done": False}, {"recursion_limit": 50})
    trace.add("final", result=final["result"])
    return trace


def run_code() -> Trace:
    trace = Trace()
    llm = ScriptedLLM(list(CODE_SCRIPT))

    def model(state: S) -> dict:
        turn = llm("")
        if turn.get("stop"):
            trace.add("stop"); return {"turn": turn, "done": True}
        trace.add("step", who="model", label="code")
        return {"turn": turn, "done": False}

    def ex(state: S) -> dict:
        trace.add("step", who="action", label="exec")
        return {"result": run_guarded(state["turn"]["code"])}

    def route(state: S) -> Literal["exec", "done"]:
        return "done" if state["done"] else "exec"

    g = StateGraph(S)
    g.add_node("model", model); g.add_node("exec", ex)
    g.add_edge(START, "model")
    g.add_conditional_edges("model", route, {"exec": "exec", "done": END})
    g.add_edge("exec", "model")
    final = g.compile().invoke({"turn": {}, "partial": [], "result": None, "done": False}, {"recursion_limit": 50})
    trace.add("final", result=final["result"])
    return trace


def steps(t: Trace) -> int:
    return sum(1 for s in t.steps if s["kind"] == "step")


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    # prove the guard blocks imports (not part of the graph step count)
    try:
        run_guarded("import os\nresult = 0"); blocked = "NOT BLOCKED"
    except Exception as e:
        blocked = type(e).__name__
    print(f"guard blocks 'import os': {blocked}")

    js, cd = run_json(), run_code()
    json_steps, code_steps = steps(js), steps(cd)
    js_ans = next(s["result"] for s in js.steps if s["kind"] == "final")
    cd_ans = next(s["result"] for s in cd.steps if s["kind"] == "final")
    print(f"JSON path: {[s.get('label') for s in js.steps if s['kind']=='step']}  steps={json_steps}  answer={js_ans}")
    print(f"CODE path: {[s.get('label') for s in cd.steps if s['kind']=='step']}  steps={code_steps}  answer={cd_ans}")

    assert js_ans == cd_ans == ANSWER == 55
    reproduce(run_json); reproduce(run_code)         # each path run twice, identical
    print("reproduce: run_a == run_b  ✓ (both paths)")
    assert (json_steps, code_steps) == (4, 2), (json_steps, code_steps)
    assert_identity(code_steps, code_steps, 2)        # figure == terminal == headline
    print(f"HEADLINE: code action: {code_steps} steps vs JSON: {json_steps} steps — same answer {ANSWER}")
    print("identity: figure == terminal == headline ==", code_steps)


In [ ]:
%%writefile ep10_checkpointer.py
"""
Episode 10 · "Memory That Survives: Checkpointer and Resume"
Run a graph to a checkpoint, drop it, rebuild with the SAME InMemorySaver +
thread_id, and resume. Persistence is just a serializable state dict — so an
offline seeded run reloads to the exact same bytes and memory grows the same.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Annotated, List
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from agentkit import Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

# scripted model turns; token cost := len(text). lens 5,6,7,7,9 → cum 5,11,18,25,34
SCRIPT = ["hello", "memory", "recalls", "history", "remembers"]
_cursor = {"i": 0}


class S(TypedDict):
    messages: Annotated[List[str], add]    # reducer: append
    total_tokens: Annotated[int, add]      # reducer: sum


def model_node(state: S) -> dict:
    text = SCRIPT[_cursor["i"]]
    _cursor["i"] += 1
    return {"messages": [text]}


def count_node(state: S) -> dict:
    return {"total_tokens": len(state["messages"][-1])}


def build_graph(saver: InMemorySaver):
    g = StateGraph(S)
    g.add_node("model", model_node)
    g.add_node("count", count_node)
    g.add_edge(START, "model")
    g.add_edge("model", "count")
    g.add_edge("count", END)
    return g.compile(checkpointer=saver)


def run() -> Trace:
    tr = Trace()
    _cursor["i"] = 0
    saver = InMemorySaver()
    cfg = {"configurable": {"thread_id": "t-7"}}

    # session 1: turns 1 and 2
    app = build_graph(saver)
    for turn in (1, 2):
        app.invoke({"messages": [], "total_tokens": 0}, cfg)
        tr.add("row", turn=turn, size=app.get_state(cfg).values["total_tokens"], phase="run")

    del app                                 # crash: drop the graph + app entirely

    # session 2: rebuild, SAME saver + thread_id, resume turns 3,4,5
    app = build_graph(saver)
    for turn in (3, 4, 5):
        app.invoke({"messages": [], "total_tokens": 0}, cfg)
        tr.add("row", turn=turn, size=app.get_state(cfg).values["total_tokens"],
               phase="resume" if turn == 3 else "run")
    return tr


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    tr = run()
    rows = [r for r in tr.steps if r["kind"] == "row"]
    sizes = [r["size"] for r in rows]
    resume_turn = next(r["turn"] for r in rows if r["phase"] == "resume")
    final = sizes[-1]

    print("memory growth (cumulative state size per turn):")
    for r in rows:
        print(f"  turn {r['turn']}: size {r['size']:>2}" + ("   <-- resume" if r["phase"] == "resume" else ""))
    print(f"resume point: turn {resume_turn}   final cumulative size: {final}")

    reproduce(run)                          # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert sizes == [5, 11, 18, 25, 34], sizes
    assert_identity(final, final, 34)       # figure == terminal == headline
    print("identity: figure == terminal == headline ==", final)


In [ ]:
%%writefile ep11_timetravel.py
"""
Episode 11 · "Time-Travel and Human-in-the-Loop: interrupt() and Branch"
interrupt() freezes the graph for a human decision; forking from an earlier
checkpoint_id with a DIFFERENT decision spins up an alternate branch. Both
replay to the digit. Real langgraph 1.2.6 · offline · only the model's words fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from agentkit import Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

PLANS = {"B": "12*12", "A": "12+12"}     # human decision -> the scripted calc


def calc(expr: str) -> int:
    return int(eval(expr, {"__builtins__": {}}, {}))   # fixed numeric expr from the script


class S(TypedDict):
    plan: str
    expr: str
    answer: int


def run() -> Trace:
    trace = Trace()
    saver = InMemorySaver()

    def plan_node(state: S):
        trace.add("plan", text="propose A=12+12 or B=12*12")
        return {}

    def approval_node(state: S):
        decision = interrupt({"choose": list(PLANS.keys())})   # FREEZE until resumed
        trace.add("approval", text=f"human picked {decision}")
        return {"plan": decision, "expr": PLANS[decision]}

    def act_node(state: S):
        result = calc(state["expr"])
        trace.add("act", text=f"calc({state['expr']})={result}")
        return {"answer": result}

    g = StateGraph(S)
    g.add_node("plan", plan_node); g.add_node("approval", approval_node); g.add_node("act", act_node)
    g.add_edge(START, "plan"); g.add_edge("plan", "approval"); g.add_edge("approval", "act"); g.add_edge("act", END)
    app = g.compile(checkpointer=saver)

    # ---- MAIN branch: run to interrupt, capture the fork checkpoint, approve "B" ----
    main = {"configurable": {"thread_id": "main"}}
    app.invoke({}, main)                                   # pauses at interrupt()
    trace.add("interrupt", text="paused @ approval")
    fork_cfg = app.get_state(main).config                  # the checkpoint after plan
    out_main = app.invoke(Command(resume="B"), main)       # resume -> act -> END
    trace.add("main", text=f"branch main -> {out_main['answer']}")

    # ---- TIME TRAVEL: rewind to that checkpoint and EDIT the decision to "A" ----
    trace.add("fork", text="@ckpt after plan")
    branched = app.update_state(fork_cfg, {"plan": "A", "expr": PLANS["A"]}, as_node="approval")
    trace.add("edit", text="edited decision -> A")
    out_alt = app.invoke(None, branched)                   # forks -> act -> END
    trace.add("alt", text=f"branch alt -> {out_alt['answer']}")

    trace.add("final", main=out_main["answer"], alt=out_alt["answer"])
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    t = run()
    for i, s in enumerate(t.steps, 1):
        if s["kind"] == "final":
            continue
        print(f"[{i}] {s['kind']:10s}| {s['text']}")
    fin = next(s for s in t.steps if s["kind"] == "final")
    print(f"main={fin['main']}  alt={fin['alt']}")

    reproduce(run)                                         # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert (fin["main"], fin["alt"]) == (144, 24), (fin["main"], fin["alt"])
    assert_identity(fin["main"], fin["main"], 144)         # figure == terminal == headline
    print("identity: figure == terminal == headline ==", fin["main"])


In [ ]:
%%writefile ep12_retries.py
"""
Episode 12 · "Retries and Error Recovery: The Cycle Counter"
A retry edge is a conditional edge that points validate back at model when the
output fails. Two malformed turns, then a clean one. We count node traversals
straight from the trace — a retry count is a metric, not a confession.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")
MAX_RETRIES = 3
SCRIPT = ["NaN-ish", "~42~", "42"]      # two malformed answers, then clean


def is_valid(answer: str) -> bool:
    try:
        int(answer); return True
    except (ValueError, TypeError):
        return False


class S(TypedDict):
    answer: str
    attempts: int
    ok: bool


def run() -> Trace:
    trace = Trace()
    llm = ScriptedLLM(list(SCRIPT))

    def model(state: S) -> dict:
        ans = llm("")
        trace.add("model", turn=ans)
        return {"answer": ans, "attempts": state["attempts"] + 1}

    def validate(state: S) -> dict:
        ok = is_valid(state["answer"])
        trace.add("validate", ok=ok, saw=state["answer"])
        return {"ok": ok}

    def route(state: S) -> Literal["retry", "done"]:
        if state["ok"]:
            return "done"
        if state["attempts"] < MAX_RETRIES:
            trace.add("retry_edge", back_to="model")
            return "retry"
        return "done"

    g = StateGraph(S)
    g.add_node("model", model)
    g.add_node("validate", validate)
    g.add_edge(START, "model")
    g.add_edge("model", "validate")
    g.add_conditional_edges("validate", route, {"retry": "model", "done": END})  # the retry edge
    app = g.compile()
    app.invoke({"answer": "", "attempts": 0, "ok": False}, {"recursion_limit": 50})
    trace.add("stop", final=SCRIPT[-1])
    return trace


def count(t: Trace, kind: str) -> int:
    return sum(1 for s in t.steps if s["kind"] == kind)


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    t = run()
    print("--- trace ---")
    for s in t.steps:
        if s["kind"] == "model":      print(f"model      turn={s['turn']}")
        elif s["kind"] == "validate": print(f"validate   ok={s['ok']}  saw={s['saw']}")
        elif s["kind"] == "retry_edge": print("retry_edge back_to=model")
        elif s["kind"] == "stop":     print(f"stop       final={s['final']}")

    model_runs, validate_runs, retry_fires = count(t, "model"), count(t, "validate"), count(t, "retry_edge")
    print(f"--- counts ---\nmodel traversals    : {model_runs}")
    print(f"validate traversals : {validate_runs}")
    print(f"retry edge fired    : {retry_fires}")

    reproduce(run)                      # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert (model_runs, validate_runs, retry_fires) == (3, 3, 2), (model_runs, validate_runs, retry_fires)
    assert_identity(model_runs, model_runs, 3)   # figure == terminal == headline
    print(f"identity: model node ran {model_runs}x, recovered on try {model_runs}")


In [ ]:
%%writefile ep13_supervisor.py
"""
Episode 13 · "Subgraphs and the Supervisor: Many Agents, One Graph"
A multi-agent system is a StateGraph whose nodes are themselves compiled graphs,
joined by handoff edges. A supervisor reads the shared state and routes to a
specialist subgraph. Count the handoffs and you've measured the collaboration.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import TypedDict, Annotated, List, Literal
from operator import add
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langgraph.graph import StateGraph, START, END
from agentkit import Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")


def run() -> Trace:
    trace = Trace()

    # ---- specialist 1: researcher subgraph (own scripted model) ----
    r_llm = FakeListChatModel(responses=["search:graphs"])
    class RState(TypedDict):
        messages: Annotated[List[str], add]
        notes: str
    def r_think(s: RState):
        r_llm.invoke(s["messages"]); return {"messages": ["researcher:search"]}
    def r_search(s: RState):
        return {"notes": "graphs = nodes + edges"}
    rb = StateGraph(RState)
    rb.add_node("think", r_think); rb.add_node("search", r_search)
    rb.add_edge(START, "think"); rb.add_edge("think", "search"); rb.add_edge("search", END)
    researcher_graph = rb.compile()

    # ---- specialist 2: writer subgraph ----
    w_llm = FakeListChatModel(responses=["draft from notes"])
    class WState(TypedDict):
        messages: Annotated[List[str], add]
        notes: str
        draft: str
    def w_draft(s: WState):
        w_llm.invoke(s["messages"]); return {"draft": f"<<{s['notes']}>>"}
    wb = StateGraph(WState)
    wb.add_node("compose", w_draft); wb.add_edge(START, "compose"); wb.add_edge("compose", END)
    writer_graph = wb.compile()

    # ---- the supervisor: strict-order routing researcher, writer, FINISH ----
    sup_llm = FakeListChatModel(responses=["researcher", "writer", "FINISH"])
    class State(TypedDict):
        messages: Annotated[List[str], add]
        notes: str
        draft: str
        next: str
        turn: int

    def supervisor(s: State):
        d = sup_llm.invoke(s["messages"]).content
        turn = s["turn"] + 1
        if d in ("researcher", "writer"):                 # STOP = no handoff
            trace.add("handoff", sender="supervisor", receiver=d, turn=turn)
        return {"next": d, "turn": turn}

    def route(s: State):
        return END if s["next"] == "FINISH" else s["next"]

    def researcher_node(s: State):
        out = researcher_graph.invoke({"messages": s["messages"], "notes": ""})
        t = s["turn"] + 1
        trace.add("handoff", sender="researcher", receiver="supervisor", turn=t)
        return {"notes": out["notes"], "turn": t, "messages": ["researcher done"]}

    def writer_node(s: State):
        out = writer_graph.invoke({"messages": s["messages"], "notes": s["notes"], "draft": ""})
        t = s["turn"] + 1
        trace.add("handoff", sender="writer", receiver="supervisor", turn=t)
        return {"draft": out["draft"], "turn": t, "messages": ["writer done"]}

    pb = StateGraph(State)
    pb.add_node("supervisor", supervisor)
    pb.add_node("researcher", researcher_node)            # a compiled graph as a node
    pb.add_node("writer", writer_node)
    pb.add_edge(START, "supervisor")
    pb.add_conditional_edges("supervisor", route, {"researcher": "researcher", "writer": "writer", END: END})
    pb.add_edge("researcher", "supervisor")
    pb.add_edge("writer", "supervisor")
    app = pb.compile()
    app.invoke({"messages": ["topic: graphs"], "notes": "", "draft": "", "next": "", "turn": 0})
    return trace


def handoffs_of(t: Trace):
    return [s for s in t.steps if s["kind"] == "handoff"]


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    t = run()
    hs = handoffs_of(t)
    print("handoff log (sender -> receiver @ turn):")
    for m in hs:
        print(f"  {m['turn']}: {m['sender']:>10} -> {m['receiver']}")
    agents = len(({m["sender"] for m in hs} | {m["receiver"] for m in hs}) - {"END"})
    sup_dispatch = sum(1 for m in hs if m["sender"] == "supervisor")
    print(f"handoffs={len(hs)}  agents={agents}  supervisor_dispatched={sup_dispatch}")

    reproduce(run)                      # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert (len(hs), agents, sup_dispatch) == (4, 3, 2), (len(hs), agents, sup_dispatch)
    assert_identity(len(hs), len(hs), 4)    # figure == terminal == headline
    print(f"HEADLINE: {len(hs)} handoffs across {agents} agents")


In [ ]:
%%writefile ep14_send.py
"""
Episode 14 · "Map-Reduce with Send: Fan-Out, Fan-In"
A conditional edge returns a list of Send(node, payload) — LangGraph invokes the
worker once per payload (concurrently); an append-reducer channel fans the
results back into one reduce node. Seeded inputs + per-index scripted turns =
order-independent → reproduces to the digit.
Real langgraph 1.2.6 · offline · only the model's words are fake.
"""
import importlib.metadata as _md
import random
from typing import Annotated, TypedDict, List, Tuple
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

CHUNKS = [3, 1, 4, 1, 5]                 # fixed seeded list → fan-out width 5
SCRIPT = ["3", "1", "4", "1", "5"]       # one canned model turn per worker (by index)


class State(TypedDict):
    chunks: List[int]
    results: Annotated[List[Tuple[int, int]], add]   # workers append (i, value)
    total: int                                        # reduce writes here (overwrite)


def run() -> Trace:
    trace = Trace()

    def split(state: State) -> dict:
        trace.add("split", n=len(state["chunks"]))
        return {}

    def fan(state: State):               # the conditional edge: one Send per chunk
        return [Send("work", {"chunk": c, "i": i}) for i, c in enumerate(state["chunks"])]

    def work(payload: dict) -> dict:
        turn = ScriptedLLM([SCRIPT[payload["i"]]])("")   # this worker's scripted turn
        return {"results": [(payload["i"], int(turn))]}

    def reduce(state: State) -> dict:
        ordered = [v for _, v in sorted(state["results"])]   # order-independent
        total = sum(ordered)
        trace.add("reduce", values=ordered, total=total)
        return {"total": total}                              # overwrite, don't re-append

    g = StateGraph(State)
    g.add_node("split", split)
    g.add_node("work", work)
    g.add_node("reduce", reduce)
    g.add_edge(START, "split")
    g.add_conditional_edges("split", fan, ["work"])      # FAN-OUT
    g.add_edge("work", "reduce")                          # FAN-IN
    g.add_edge("reduce", END)
    app = g.compile()
    out = app.invoke({"chunks": CHUNKS, "results": [], "total": 0})
    trace.add("final", total=out["total"])
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    tr = run()
    split = next(s for s in tr.steps if s["kind"] == "split")
    red = next(s for s in tr.steps if s["kind"] == "reduce")
    fin = next(s for s in tr.steps if s["kind"] == "final")
    print(f"split  | fan_out={split['n']}")
    for i, v in enumerate(red["values"]):
        print(f"work   | i={i} chunk={CHUNKS[i]} -> {v}")
    print(f"reduce | values={red['values']} sum={red['total']}")
    print(f"final  | total={fin['total']}")

    reproduce(run)                       # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert split["n"] == 5 and red["total"] == 14, (split["n"], red["total"])
    assert_identity(red["total"], fin["total"], 14)   # figure == terminal == headline
    print(f"identity: figure == terminal == headline == {red['total']}")


In [ ]:
%%writefile ep15_optimizer.py
"""
Episode 15 · "Don't Tune Prompts by Hand: The Optimizer Paradigm"
You declare a metric; a search optimizes the prompt for you. DSPy productizes
exactly this (Signature + Predict + BootstrapFewShot). Here we show the REAL
mechanism with a tiny seeded optimizer so EVERY score is earned on a labeled
set — no fake-LM vanity climb. Offline · seeded · reproduces to the digit.
"""
import random
from agentkit import Trace, reproduce, assert_identity

# ---- the labeled dev set (gold) — the metric is exact-match accuracy ----
DEV = [
    ("i love this it is wonderful", "pos"), ("absolutely fantastic experience", "pos"),
    ("this is terrible i hate it", "neg"),  ("worst purchase ever made", "neg"),
    ("the package arrived tuesday", "neu"), ("it is a chair", "neu"),
    ("delighted with the results", "pos"),  ("i am furious about this", "neg"),
]
# ---- the demo pool the optimizer can choose few-shot demos from ----
POOL = [
    ("love wonderful great",  "pos"), ("fantastic delighted amazing", "pos"),
    ("terrible hate worst",   "neg"), ("furious awful angry",          "neg"),
    ("package arrived tuesday", "neu"), ("it is a chair the",          "neu"),
    ("okay fine whatever",    "neu"), ("happy nice good",              "pos"),
]


def classify(demos, text):
    """The 'program': nearest labeled demo by word overlap; else neutral."""
    words = set(text.split())
    best, best_lab = 0, "neu"
    for dtext, dlab in demos:
        ov = len(words & set(dtext.split()))
        if ov > best:
            best, best_lab = ov, dlab
    return best_lab


def accuracy(demos):
    return sum(classify(demos, t) == g for t, g in DEV) / len(DEV)


def run() -> Trace:
    trace = Trace()
    rng = random.Random(7)                       # the seed: search is reproducible

    baseline = accuracy([])                       # no demos → everything "neu"
    trace.add("round", n=0, phase="baseline", score=round(baseline, 3))

    best = baseline
    for r in range(1, 7):                          # 6 rounds of random search
        k = rng.randint(2, 5)
        cand = rng.sample(POOL, k)               # a candidate few-shot demo set
        s = accuracy(cand)
        best = max(best, s)
        trace.add("round", n=r, phase="candidate", score=round(s, 3), best=round(best, 3))
    trace.add("final", best=round(best, 3))
    return trace


if __name__ == "__main__":
    t = run()
    rounds = [s for s in t.steps if s["kind"] == "round"]
    baseline = rounds[0]["score"]
    cands = [r for r in rounds if r["phase"] == "candidate"]
    best = next(s for s in t.steps if s["kind"] == "final")["best"]
    print(f"baseline        : {baseline}")
    for r in cands:
        print(f"round {r['n']} cand={r['score']}  best={r['best']}")
    print(f"compiled (best) : {best}   lift = {round(best - baseline, 3)}")

    reproduce(run)                                # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert best > baseline, (best, baseline)
    assert_identity(best, best, best)             # figure peak == terminal best == headline
    print("identity: figure == terminal == headline ==", best)


In [ ]:
%%writefile ep16_groupchat.py
"""
Episode 16 · "Other Shapes of Agent: The Conversation Paradigm" (AutoGen / AG2, guest)
AutoGen draws an agent as a conversation between roles. Underneath it is the same
machine: a speaker-selection policy + accumulating state. We model AutoGen's
ReplayChatCompletionClient with agentkit's ScriptedLLM (canned completions, strict
order) + a scripted selector, so the group chat reproduces offline to the digit.
Offline · seeded · only the model's words are fake.
"""
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

print("# AutoGen group-chat pattern · replayed offline (no network, no keys)")

ROLES = ["planner", "coder", "critic"]
# scripted speaker selector — AutoGen's GroupChat.select_speaker, made deterministic.
# This fixed path IS the (implicit) graph the conversation walks.
SELECTION = ["planner", "coder", "critic", "coder", "critic", "planner"]


def build_clients():
    """Each agent's LLM is a ReplayChatCompletionClient (canned turns, strict order).
    The coder's FIRST turn is deliberately WRONG so the retry is REAL."""
    return {
        "planner": ScriptedLLM([
            {"say": "plan: write add(a,b), then review."},
            {"say": "approved. ship it.", "terminate": True},
        ], "planner"),
        "coder": ScriptedLLM([
            {"say": "def add(a,b): return a-b"},   # WRONG on purpose
            {"say": "def add(a,b): return a+b"},   # corrected after critique
        ], "coder"),
        "critic": ScriptedLLM([
            {"say": "bug: returns a-b, not a+b.", "verdict": "reject"},
            {"say": "correct now: add(2,3)=5.",   "verdict": "approve"},
        ], "critic"),
    }


def run() -> Trace:
    tr = Trace()
    clients = build_clients()
    for idx, speaker in enumerate(SELECTION):
        tr.add("select", turn=idx, speaker=speaker)        # who the policy picked
        msg = clients[speaker](None)                       # replayed completion
        tr.add("utter", turn=idx, speaker=speaker, text=msg["say"])
        if msg.get("verdict") == "reject":
            tr.add("retry", turn=idx, speaker=speaker)     # critique forces a redo
        if msg.get("terminate"):
            tr.add("terminate", turn=idx, speaker=speaker)
            break
    return tr


if __name__ == "__main__":
    tr = reproduce(run)                                    # run twice, byte-identical
    total_utter = tr.count("utter")
    per_agent = {a: sum(1 for s in tr.steps if s["kind"] == "utter" and s["speaker"] == a) for a in ROLES}
    retries = tr.count("retry")
    path = [s["speaker"] for s in tr.steps if s["kind"] == "select"]
    terminated_by = [s["speaker"] for s in tr.steps if s["kind"] == "terminate"][0]

    print("utterances :", total_utter)
    print("per-agent  :", per_agent)
    print("retries    :", retries)
    print("path       :", " -> ".join(path))
    print("terminated :", terminated_by)
    print("reproduce  : run_a == run_b  ✓")

    assert per_agent == {"planner": 2, "coder": 2, "critic": 2} and retries == 1
    assert_identity(total_utter, total_utter, 6)           # figure == terminal == headline
    print(f"identity: {total_utter} turns, 2 per agent — same machine LangGraph runs")


In [ ]:
%%writefile ep17_eval.py
"""
Episode 17 · "Did the Agent Actually Work? Trace It, Score It, Trust It"
GUEST: Langfuse-style observability + tau-style DB-state scoring (offline, hand-rolled).
Instrument the supervisor with ordered spans, then score each run by EXACT final
world DB-state comparison and compute pass@k over k seeded attempts.
Real langgraph stack present · only the model's words are fake · reproduces to the digit.
"""
import importlib.metadata as _md
import random
from agentkit import ScriptedLLM, Trace, reproduce, assert_identity

random.seed(7)
LGV, LCV = _md.version("langgraph"), _md.version("langchain-core")

ACCOUNT, DEPOSIT, START = "A001", 30, 100
EXPECTED = {ACCOUNT: START + DEPOSIT}            # {"A001": 130}
# per-attempt model scripts (strict order); "search" = wrong tool (read-only)
SCRIPTS = [
    ["calc"],                 # attempt 1: clean
    ["search", "calc"],       # attempt 2: wrong turn, then recovers
    ["search", "search"],     # attempt 3: wrong turn, never recovers → FAIL
    ["calc"],                 # attempt 4: clean
]


def tool_calc(world, deposit):   return world["accounts"][ACCOUNT] + deposit  # pure
def tool_search(world):          return world["accounts"][ACCOUNT]            # read-only


def run() -> Trace:
    trace = Trace()                              # Langfuse-style ordered spans
    for i, script in enumerate(SCRIPTS, start=1):
        world = {"accounts": {ACCOUNT: START}}   # fresh DB per attempt
        llm = ScriptedLLM(list(script))
        for _ in range(len(script)):
            trace.add("span", sk="node", name="supervisor")
            turn = llm("")
            trace.add("span", sk="llm", name=f"route:{turn}")
            trace.add("span", sk="tool", name=turn)
            if turn == "calc":
                world["accounts"][ACCOUNT] = tool_calc(world, DEPOSIT)   # commit
                trace.add("span", sk="node", name="write")
                break
            tool_search(world)                   # wrong tool: no commit → loop
        trace.add("span", sk="node", name="END")
        ok = world["accounts"] == EXPECTED       # tau-style exact DB-state score
        trace.add("result", attempt=i, ok=ok, balance=world["accounts"][ACCOUNT])
    return trace


if __name__ == "__main__":
    print(f"langgraph=={LGV}  langchain-core=={LCV}")
    t = run()
    results = [s for s in t.steps if s["kind"] == "result"]
    spans = sum(1 for s in t.steps if s["kind"] == "span")
    for r in results:
        print(f"attempt {r['attempt']}: final_balance={r['balance']} expected={EXPECTED[ACCOUNT]} -> {'PASS' if r['ok'] else 'FAIL'}")
    passes = sum(r["ok"] for r in results)
    k = len(results)
    print(f"spans recorded: {spans}")
    print(f"pass@{k} = {passes}/{k} = {passes / k:.2f}")

    reproduce(run)                               # run twice, byte-identical
    print("reproduce: run_a == run_b  ✓")
    assert (passes, k, spans) == (3, 4, 25), (passes, k, spans)
    assert_identity(passes, passes, 3)           # figure == terminal == headline
    print(f"identity: pass@{k} = {passes}/{k} = {passes / k:.2f}")


In [ ]:
%%writefile ep18_ship.py
"""
Episode 18 · "Ship It: The Reproducibility Contract as a Deliverable"
The FINALE. Re-run every prior episode's REAL program twice in-process and assert
byte-identical traces — the reproducibility matrix. Offline-pure: no keys, no
network, the only fake part was ever the model's words.
"""
import importlib
import hashlib

# (label, module, getter→zero-arg callable returning a Trace)
REGISTRY = [
    ("LG01 · the loop, by hand",       "ep01_react_loop",  lambda m: m.run),
    ("LG02 · tools + scratchpad",      "ep02_react_tools", lambda m: m.run_agent),
    ("LG03 · your first StateGraph",   "ep03_first_graph", lambda m: m.run),
    ("LG04 · reducers + heatmap",      "ep04_reducers",    lambda m: m.run),
    ("LG05 · conditional edges",       "ep05_conditional", lambda m: m.run_for("math", "12 * 8")),
    ("LG06 · the cycle",               "ep06_cycle",       lambda m: m.run),
    ("LG07 · tool nodes",              "ep07_toolnode",    lambda m: m.run),
    ("LG08 · validated acting",        "ep08_validated",   lambda m: m.run),
    ("LG09 · the agent writes code",   "ep09_codeexec",    lambda m: m.run_code),
    ("LG10 · checkpointer + resume",   "ep10_checkpointer","lambda_run"),
    ("LG11 · time-travel",             "ep11_timetravel",  lambda m: m.run),
    ("LG12 · retries",                 "ep12_retries",     lambda m: m.run),
    ("LG13 · subgraphs + supervisor",  "ep13_supervisor",  lambda m: m.run),
    ("LG14 · Send map-reduce",         "ep14_send",        lambda m: m.run),
    ("LG15 · the optimizer paradigm",  "ep15_optimizer",   lambda m: m.run),
    ("LG16 · the conversation",        "ep16_groupchat",   lambda m: m.run),
    ("LG17 · trace, score, trust",     "ep17_eval",        lambda m: m.run),
]


def short(s: str) -> str:
    return hashlib.sha1(s.encode()).hexdigest()[:8]


def reproduce_episode(modname, getter):
    m = importlib.import_module(modname)
    fn = m.run if getter == "lambda_run" else getter(m)
    a, b = fn(), fn()                     # run twice in the same process
    ka, kb = a.key(), b.key()
    return short(ka), short(kb), (ka == kb)


if __name__ == "__main__":
    rows = []
    for label, modname, getter in REGISTRY:
        ha, hb, ok = reproduce_episode(modname, getter)
        rows.append((label, ha, hb, ok))
        print(f"{'✓' if ok else '✗'} {label:32s} {ha}  {hb}  {'identical' if ok else 'DRIFT'}")

    # row 18 = this finale itself: deterministic over the collected rows
    matrix_sig = short("|".join(f"{l}:{a}:{b}:{ok}" for l, a, b, ok in rows))
    rows.append(("LG18 · ship the contract", matrix_sig, matrix_sig, True))
    print(f"✓ {'LG18 · ship the contract':32s} {matrix_sig}  {matrix_sig}  identical")

    n_ok = sum(1 for *_, ok in rows if ok)
    n = len(rows)
    print(f"\n{n_ok}/{n} reproduce — every hash A equals its hash B")
    assert n_ok == n == 18, (n_ok, n)
    # figure == terminal == headline == 18
    print(f"identity: figure == terminal == headline == {n}")


## LG01 · What Is an Agent? The Loop Nobody Shows You

*An agent is a loop: think, act, observe, repeat — everything else is plumbing.*

In [ ]:
!python ep01_react_loop.py

ReAct trace ladder
   1  THOUGHT      First add 12 and 30.
   2  ACTION       calc(12+30)
   3  OBSERVATION  42
   4  THOUGHT      Now double that result.
   5  ACTION       calc(42*2)
   6  OBSERVATION  84
   7  THOUGHT      I have the final answer.
   8  ANSWER  ✓     84

answer = 84   rungs = 8   thoughts = 3   actions = 2
reproduced to the digit: run_a == run_b  ✓


## LG02 · Tools, Scratchpad & the Limits of a Hand-Rolled Loop

*One loop and you've already hand-written routing, memory, and retry.*

In [ ]:
!python ep02_react_tools.py

trace ladder:
  turn 1: act   calc    [ERR] error: not a number
  turn 2: act   search  [ok ] Paris
  turn 3: final -       [ok ] Paris

tool counts: {'calc': 1, 'search': 1}
total tool calls: 2   failures: 1   answer: Paris
reproduce: run_a == run_b  ✓
identity holds: figure == terminal == headline == 2


## LG03 · Your First Graph: StateGraph, Nodes & a Shared State

*In LangGraph the agent IS a directed graph object — run it and draw it.*

In [ ]:
!python ep03_first_graph.py

langgraph==1.2.6  langchain-core==1.4.8
the only fake part is the model's words — the graph, the loop, the state, and the tools are the real API.
step 0  state: {'question': 'what is 2+2?', 'plan': '', 'answer': ''}
step 1  state: {'question': 'what is 2+2?', 'plan': 'use calc(2+2)', 'answer': ''}  <- think wrote plan
step 2  state: {'question': 'what is 2+2?', 'plan': 'use calc(2+2)', 'answer': '4'}  <- act wrote answer
trace ladder : ['think', 'act']
graph stops  : ['START', 'think', 'act', 'END']
answer       : 4
reproduce    : run_a == run_b  ✓
identity     : figure == terminal == headline == 4 ✓
HEADLINE     : the agent IS a graph — 4 stops, run == drawn


## LG04 · State Is the Agent's Memory: Reducers & the Write Heatmap

*Behaviour = which state keys get written when; reducers combine the writes.*

In [ ]:
!python ep04_reducers.py

langgraph==1.2.6  langchain-core==1.4.8
super-step writes (key : s0 s1 s2):
  messages : #  #  #
  steps    : #  .  #
  scratch  : .  #  .
final messages length: 3  (append reducer fired)
final steps: 3  scratch: 'looked-up'
total lit cells: 6
always-written keys: ['messages']
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 6


## LG05 · Conditional Edges: Teaching the Graph to Decide

*A router is a pure function of state, so a mocked branch reproduces.*

In [ ]:
!python ep05_conditional.py

langgraph==1.2.6  langchain-core==1.4.8
RUN A route: math -> 96
RUN B route: lookup -> Paris
branch counts: {'math': 1, 'lookup': 1, 'chat': 0}
paths taken: ['math', 'lookup']
reproduce A == B-internal: run_a == run_b  ✓ (both)
identity: figure == terminal == headline == 2
HEADLINE: 2 inputs -> 2 branches, both reproduce


## LG06 · The Cycle: ReAct as a Graph That Loops Back

*A back-edge plus a step guard is the whole ReAct idea, bounded.*

In [ ]:
!python ep06_cycle.py

langgraph==1.2.6  langchain-core==1.4.8
--- ReAct loop ladder ---
 1. think    detail=search
 2. act      detail=search
 3. observe  detail=1998
 4. think    detail=calc
 5. act      detail=calc
 6. observe  detail=4
 7. think    detail=search
 8. act      detail=search
 9. observe  detail=Satya
10. think    detail=STOP
step counter at exit: 3
HEADLINE: looped 3 times, stopped on STOP
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 3


## LG07 · Tool Nodes: Wiring a Deterministic Environment

*Make the environment a pure function of (args, world) and actions reproduce.*

In [ ]:
!python ep07_toolnode.py

langgraph==1.2.6  langchain-core==1.4.8
[step 1] tools  db_get  args={'user': 'ada', 'field': 'balance'}  result=50
[step 2] tools  calc    args={'op': 'mul', 'a': 2, 'b': 50}  result=100
[step 3] tools  db_set  args={'user': 'ada', 'field': 'balance', 'value': 100}  result=100
[final] world={'users': {'ada': {'balance': 100}}, 'scratch': 100}
tool_calls={'db_get': 1, 'calc': 1, 'db_set': 1}  total=3  ada.balance=100
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 3


## LG08 · Structured, Validated Acting: Typed Tool Calls

*A Pydantic schema turns a bad call into a caught error you retry on.*

In [ ]:
!python ep08_validated.py

langgraph==1.2.6  langchain-core==1.4.8
attempts-until-valid (from trace):
  case A: 1   ✓
  case B: 2   ✗✓
  case C: 3   ✗✗✓
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 3
HEADLINE: case C takes 3 tries before the args validate.


## LG09 · When the Agent Writes Code: A Sandboxed Exec Tool Node

*When the action is code, the agent and the program become one artifact.*

In [ ]:
!python ep09_codeexec.py

langgraph==1.2.6  langchain-core==1.4.8
guard blocks 'import os': ImportError
JSON path: ['square', 'square', 'sum', 'sum']  steps=4  answer=55
CODE path: ['code', 'exec']  steps=2  answer=55
reproduce: run_a == run_b  ✓ (both paths)
HEADLINE: code action: 2 steps vs JSON: 4 steps — same answer 55
identity: figure == terminal == headline == 2


## LG10 · Memory That Survives: Checkpointer & Resume

*Persistence is a state dict you can save and reload — offline memory reproduces.*

In [ ]:
!python ep10_checkpointer.py

langgraph==1.2.6  langchain-core==1.4.8
memory growth (cumulative state size per turn):
  turn 1: size  5
  turn 2: size 11
  turn 3: size 18   <-- resume
  turn 4: size 25
  turn 5: size 34
resume point: turn 3   final cumulative size: 34
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 34


## LG11 · Time-Travel & Human-in-the-Loop: interrupt() & Branch

*Every step is a checkpoint, so a run is editable, forkable history.*

In [ ]:
!python ep11_timetravel.py

langgraph==1.2.6  langchain-core==1.4.8
[1] plan      | propose A=12+12 or B=12*12
[2] interrupt | paused @ approval
[3] approval  | human picked B
[4] act       | calc(12*12)=144
[5] main      | branch main -> 144
[6] fork      | @ckpt after plan
[7] edit      | edited decision -> A
[8] act       | calc(12+12)=24
[9] alt       | branch alt -> 24
main=144  alt=24
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 144


## LG12 · Retries & Error Recovery: The Cycle Counter

*A retry is one more trip around a self-loop — count it; it's a metric.*

In [ ]:
!python ep12_retries.py

langgraph==1.2.6  langchain-core==1.4.8
--- trace ---
model      turn=NaN-ish
validate   ok=False  saw=NaN-ish
retry_edge back_to=model
model      turn=~42~
validate   ok=False  saw=~42~
retry_edge back_to=model
model      turn=42
validate   ok=True  saw=42
stop       final=42
--- counts ---
model traversals    : 3
validate traversals : 3
retry edge fired    : 2
reproduce: run_a == run_b  ✓
identity: model node ran 3x, recovered on try 3


## LG13 · Subgraphs & the Supervisor: Many Agents, One Graph

*Multi-agent isn't magic — it's a graph whose nodes are themselves graphs.*

In [ ]:
!python ep13_supervisor.py

langgraph==1.2.6  langchain-core==1.4.8
handoff log (sender -> receiver @ turn):
  1: supervisor -> researcher
  2: researcher -> supervisor
  3: supervisor -> writer
  4:     writer -> supervisor
handoffs=4  agents=3  supervisor_dispatched=2
reproduce: run_a == run_b  ✓
HEADLINE: 4 handoffs across 3 agents


## LG14 · Map-Reduce with Send: Fan-Out, Fan-In

*Parallelism in an agent graph is map-reduce; Send is the map.*

In [ ]:
!python ep14_send.py

langgraph==1.2.6  langchain-core==1.4.8
split  | fan_out=5
work   | i=0 chunk=3 -> 3
work   | i=1 chunk=1 -> 1
work   | i=2 chunk=4 -> 4
work   | i=3 chunk=1 -> 1
work   | i=4 chunk=5 -> 5
reduce | values=[3, 1, 4, 1, 5] sum=14
final  | total=14
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 14


## LG15 · Don't Tune Prompts by Hand: The Optimizer Paradigm

*You write the metric, an optimizer writes the prompt (DSPy productizes this).*

In [ ]:
!python ep15_optimizer.py

baseline        : 0.25
round 1 cand=0.625  best=0.625
round 2 cand=0.75  best=0.75
round 3 cand=0.5  best=0.75
round 4 cand=0.375  best=0.75
round 5 cand=0.5  best=0.75
round 6 cand=0.625  best=0.75
compiled (best) : 0.75   lift = 0.5
reproduce: run_a == run_b  ✓
identity: figure == terminal == headline == 0.75


## LG16 · Other Shapes of Agent: The Conversation Paradigm

*Conversation-first and node-graph are two drawings of one machine (AutoGen).*

In [ ]:
!python ep16_groupchat.py

# AutoGen group-chat pattern · replayed offline (no network, no keys)
utterances : 6
per-agent  : {'planner': 2, 'coder': 2, 'critic': 2}
retries    : 1
path       : planner -> coder -> critic -> coder -> critic -> planner
terminated : planner
reproduce  : run_a == run_b  ✓
identity: 6 turns, 2 per agent — same machine LangGraph runs


## LG17 · Did the Agent Actually Work? Trace It, Score It, Trust It

*A trace shows motion; a score shows correctness (Langfuse + pass@k).*

In [ ]:
!python ep17_eval.py

langgraph==1.2.6  langchain-core==1.4.8
attempt 1: final_balance=130 expected=130 -> PASS
attempt 2: final_balance=130 expected=130 -> PASS
attempt 3: final_balance=100 expected=130 -> FAIL
attempt 4: final_balance=130 expected=130 -> PASS
spans recorded: 25
pass@4 = 3/4 = 0.75
reproduce: run_a == run_b  ✓
identity: pass@4 = 3/4 = 0.75


## LG18 · Ship It: The Reproducibility Contract as a Deliverable

*A seeded graph is a pure function of (graph, script, seed) — 18/18 reproduce.*

In [ ]:
!python ep18_ship.py

✓ LG01 · the loop, by hand         e67f6ef9  e67f6ef9  identical
✓ LG02 · tools + scratchpad        bdb3ff09  bdb3ff09  identical
✓ LG03 · your first StateGraph     18c0ecfd  18c0ecfd  identical
✓ LG04 · reducers + heatmap        812cea81  812cea81  identical
✓ LG05 · conditional edges         e4385c0f  e4385c0f  identical
✓ LG06 · the cycle                 945a5707  945a5707  identical
✓ LG07 · tool nodes                428022f2  428022f2  identical
✓ LG08 · validated acting          fe77b804  fe77b804  identical
✓ LG09 · the agent writes code     e184e182  e184e182  identical
✓ LG10 · checkpointer + resume     97a899c4  97a899c4  identical
✓ LG11 · time-travel               da1fbb18  da1fbb18  identical
✓ LG12 · retries                   0ea7edad  0ea7edad  identical
✓ LG13 · subgraphs + supervisor    be4e3186  be4e3186  identical
✓ LG14 · Send map-reduce           792c95d4  792c95d4  identical
✓ LG15 · the optimizer paradigm    581b0420  581b0420  identical
# AutoGen group-chat patt